<a href="https://colab.research.google.com/github/mayagrita/Panoramic_Dent_AI/blob/marla/Copy_of_Panoramic_dent_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
import os
import cv2
import numpy as np
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Paths
dataset_path = '/content/drive/MyDrive/dental_dataset/dental_dataset'
images_folder = os.path.join(dataset_path, "images", "train")
# images_folder = os.path.join(dataset_path, "images", "valid")

output_csv = os.path.join(dataset_path, "manual_features_train2.csv")

# Check if the folder exists
if not os.path.exists(images_folder):
    raise FileNotFoundError(f"Image folder not found: {images_folder}")

# List of image files
image_files = [f for f in os.listdir(images_folder) if f.lower().endswith((".jpg", ".png"))]
print(f"Number of images found: {len(image_files)}")

def extract_manual_features(img_path):
    """
    Extract 5 statistical features from a medical image.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Could not read image: {img_path}")

    features = {}

    # 1. Mean intensity
    features["mean_intensity"] = float(np.mean(img))

    # 2. Standard deviation of intensity
    features["std_intensity"] = float(np.std(img))

    # 3. Number of dark pixels
    dark_pixels = np.sum(img < np.percentile(img, 5))
    features["dark_pixel_count"] = int(dark_pixels)

    # 4. Symmetry score between left and right halves
    left_half = img[:, :img.shape[1] // 2]
    right_half = img[:, img.shape[1] // 2:]
    symmetry_score = abs(np.mean(left_half) - np.mean(right_half))
    features["symmetry_score"] = float(symmetry_score)

    # 5. Image Entropy (randomness in the image)
    hist, _ = np.histogram(img.flatten(), bins=256, range=(0, 256))
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-8))
    features["image_entropy"] = float(entropy)

    return features

# Extract features from all images
features_list = []

for i, img_file in enumerate(image_files):
    print(f"[{i+1}/{len(image_files)}] Processing: {img_file}")
    try:
        img_path = os.path.join(images_folder, img_file)
        features = extract_manual_features(img_path)
        features["image_name"] = img_file
        features_list.append(features)
    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")
        continue

# Convert to DataFrame and save to CSV
df = pd.DataFrame(features_list)
df.to_csv(output_csv, index=False)

print(f"\nFeatures saved to: {output_csv}")
print("First 5 rows:")
print(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of images found: 2871
[1/2871] Processing: 4066680000-jpg_png_jpg.rf.6451cc4f90887ed6f073de71ff8e3eb1.jpg
[2/2871] Processing: 4066680000-jpg_png_jpg.rf.bb43289ee2e415a43f8ed2c7344be1cd.jpg
[3/2871] Processing: 4066870000-jpg_png_jpg.rf.f4de549d54ac8190167da0ff54317d08.jpg
[4/2871] Processing: 4067600000-jpg_png_jpg.rf.6599fb9629ccb11f404a2800df54cd64.jpg
[5/2871] Processing: 4067600000-jpg_png_jpg.rf.9f46e7fe7e303a54b9014d14da83422b.jpg
[6/2871] Processing: 4068250000-jpg_png_jpg.rf.6212b5f121df626658c0980ce231c225.jpg
[7/2871] Processing: 4068530000-jpg_png_jpg.rf.a96f4733d190df1061e0e1e35a19b4a3.jpg
[8/2871] Processing: 4068540000-jpg_png_jpg.rf.594a9aefa49e696a19238187ea3b4ecf.jpg
[9/2871] Processing: 4068540000-jpg_png_jpg.rf.de1766ccd853f0557fd2d77f14a39362.jpg
[10/2871] Processing: 4069100000-jpg_png_jpg.rf.35bf35ed7c6c16e7df23c79dca03fad5.jpg
[

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dental_dataset/dental_dataset/manual_features_train2.csv")
print(" عدد الصفوف:", len(df))
print(df.head(10))
print(" الأعمدة الموجودة:")
print(df.columns.tolist())

 عدد الصفوف: 9443
   mean_intensity  std_intensity  dark_pixel_count  symmetry_score  \
0       88.844697      53.557402                 0        4.106357   
1      105.238589      60.210632                 0        3.483545   
2       98.488962      57.865843                 0        3.496548   
3       96.828770      56.193228                 0        2.329219   
4       75.586833      49.533948                 0        2.316040   
5      111.527019      62.701976                 0        6.282476   
6      143.643188      60.962198              2380        6.198076   
7      109.689028      63.888676                 0        2.578789   
8      124.294851      63.221685              7102        2.592622   
9       96.858447      58.513415                 0        1.871172   

   image_entropy                                         image_name  
0       6.825686  cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf....  
1       6.740990  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  
2

In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

class CNNFeatureExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.feature_extractor = torch.nn.Sequential(*list(self.model.children())[:-1])

    def forward(self, x):
        with torch.no_grad():
            features = self.feature_extractor(x).flatten(start_dim=1)
        return features

In [ ]:
import os
images_folder = "/content/drive/MyDrive/dental_dataset/dental_dataset/images/train"
output_csv = "/content/drive/MyDrive/dental_dataset/dental_dataset/cnn_features_train.csv"

os.makedirs(os.path.dirname(output_csv), exist_ok=True)

In [ ]:
from torchvision.io import read_image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
cnn_extractor = CNNFeatureExtractor()
cnn_extractor.eval()

image_files = [f for f in os.listdir(images_folder) if f.lower().endswith((".jpg", ".png"))]
features_list = []

for i, img_file in enumerate(image_files):
    print(f"Extracting features from image {i+1}/{len(image_files)}: {img_file}")

    try:
        img_path = os.path.join(images_folder, img_file)
        img = read_image(img_path)
        img = transform(img).unsqueeze(0)  # Shape: [1, 3, 224, 224]

        with torch.no_grad():
            cnn_features = cnn_extractor(img).squeeze(0).cpu().numpy()  # Shape: [2048]

        features_list.append({
            "image_name": img_file,
            **{f"cnn_{j}": cnn_features[j] for j in range(cnn_features.shape[0])}
        })

    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")
        continue

# Save features to CSV
df_cnn = pd.DataFrame(features_list)
df_cnn.to_csv(output_csv, index=False)

print(f"CNN features saved to: {output_csv}")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]


Extracting features from image 1/9443: cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf.b69da0570af4e1412f3db4930530b05e.jpg


/usr/local/lib/python3.11/dist-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


Streaming output truncated to the last 5000 lines.
Extracting features from image 4444/9443: 3975470000-jpg_png_jpg.rf.39e54ff36fa9d84b2511649d72549353.jpg
Extracting features from image 4445/9443: 3975470000-jpg_png_jpg.rf.74efec0eebc1fa19b250b1b6d622e1f0.jpg
Extracting features from image 4446/9443: 3975470000-jpg_png_jpg.rf.ead21d51a77dc6fa2ca200eda907fcc1.jpg
Extracting features from image 4447/9443: 3975520000-jpg_png_jpg.rf.6e4fda94da07ac0b60814bb6800c5d21.jpg
Extracting features from image 4448/9443: 3975520000-jpg_png_jpg.rf.8a9facf3c4c69fe5f05f7a6079b6e31d.jpg
Extracting features from image 4449/9443: 3975520000-jpg_png_jpg.rf.ba16dd8d454b141b57f1a939fa0956c2.jpg
Extracting features from image 4450/9443: 3975720000-jpg_png_jpg.rf.33dc4d3cb37a037be1c4a4ec7cc01466.jpg
Extracting features from image 4451/9443: 3975720000-jpg_png_jpg.rf.4e454c9c8614ac7bf9fb2e6be4eacabe.jpg
Extracting features from image 4452/9443: 3975720000-jpg_png_jpg.rf.efaded0394aed816cd0ebe6387521ee2.jpg
Extr

NameError: name 'pd' is not defined

In [ ]:
import pandas as pd

# Check if there is any data
if len(features_list) > 0:
    df_cnn = pd.DataFrame(features_list)
    df_cnn.to_csv(output_csv, index=False)
    print(f"Features saved to: {output_csv}")
else:
    print("No data to save.")


Features saved to: /content/drive/MyDrive/dental_dataset/dental_dataset/cnn_features_train.csv


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dental_dataset/dental_dataset/cnn_features_train.csv")
print(" عدد الصفوف:", len(df))
print(df.head(10))
print(" الأعمدة الموجودة:")
print(df.columns.tolist())

 عدد الصفوف: 9443
                                          image_name     cnn_0     cnn_1  \
0  cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf....  0.031355  0.619478   
1  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.032178  0.493335   
2  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.034500  0.426229   
3  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.042977  0.253108   
4  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.057880  0.300909   
5  cropped_ARTI-_2023-10-26181337_1_png.rf.a9a5f0...  0.109148  0.778551   
6  cropped_ARTI-_2023-10-26181337_1_png.rf.b733f6...  0.148565  0.919087   
7  cropped_Arun-Bansal_2023-10-26155146_1_png.rf....  0.140749  0.904578   
8  cropped_Arun-Bansal_2023-10-26155146_1_png.rf....  0.142311  0.834245   
9  cropped_ARVIND-SINGH_2023-10-21135329_1_png.rf...  0.097737  1.409632   

      cnn_2     cnn_3     cnn_4     cnn_5     cnn_6     cnn_7     cnn_8  ...  \
0  0.040540  0.084724  0.599837  0.366325  0.540656  0.319573  0.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dental_dataset/combined_features.csv")
print(" عدد الصفوف:", len(df))
print(df.head(10))
print(" الأعمدة الموجودة:")
print(df.columns.tolist())

 عدد الصفوف: 9443
                                          image_name     cnn_0     cnn_1  \
0  cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf....  0.031355  0.619478   
1  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.032178  0.493335   
2  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.034500  0.426229   
3  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.042977  0.253108   
4  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.057880  0.300909   
5  cropped_ARTI-_2023-10-26181337_1_png.rf.a9a5f0...  0.109148  0.778551   
6  cropped_ARTI-_2023-10-26181337_1_png.rf.b733f6...  0.148565  0.919087   
7  cropped_Arun-Bansal_2023-10-26155146_1_png.rf....  0.140749  0.904578   
8  cropped_Arun-Bansal_2023-10-26155146_1_png.rf....  0.142311  0.834245   
9  cropped_ARVIND-SINGH_2023-10-21135329_1_png.rf...  0.097737  1.409632   

      cnn_2     cnn_3     cnn_4     cnn_5     cnn_6     cnn_7     cnn_8  ...  \
0  0.040540  0.084724  0.599837  0.366325  0.540656  0.319573  0.

In [ ]:
import json
import pandas as pd

json_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/annotations/train_coco.json"
with open(json_path, 'r') as f:
    data = json.load(f)

category_to_name = {category['id']: category['name'] for category in data['categories']}
category_ids = sorted(category_to_name.keys())

class_names = [category_to_name[cid] for cid in category_ids]

num_classes = len(class_names)

category_to_index = {cid: idx for idx, cid in enumerate(category_ids)}

images_data = []
for image in data['images']:
    image_id = image['id']
    file_name = image['file_name']

    label_vector = [0] * num_classes

    for ann in data['annotations']:
        if ann['image_id'] == image_id:
            category_id = ann['category_id']
            class_index = category_to_index[category_id]
            label_vector[class_index] = 1

    image_entry = {'image_name': file_name}
    image_entry.update({class_names[i]: label_vector[i] for i in range(num_classes)})
    images_data.append(image_entry)

df_labels = pd.DataFrame(images_data)
df_labels.columns = df_labels.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
df_labels.to_csv('/content/drive/MyDrive/dental_dataset/image_labels.csv', index=False)



In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dental_dataset/image_labels.csv")
print(" عدد الصفوف:", len(df))
print(df.head(10))
print(" الأعمدة الموجودة:")
print(df.columns.tolist())

 عدد الصفوف: 9481
                                          image_name  Caries  Crown  Filling  \
0  000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-121853...       1      1        1   
1  000dc27f-NAJIB_MARDANLOO_MASUME_2020-07-121853...       1      1        1   
2  003e5f80-MOHAMADI_ATEFEH_2020-06-01173808_jpg....       1      1        1   
3  0043783e-Asiadar_Leila_2022-06-12141026_jpg.rf...       1      1        1   
4  00cf39c1-Karaptiyan_Robert_50yo_13032021_18590...       0      1        1   
5  00cf39c1-Karaptiyan_Robert_50yo_13032021_18590...       0      1        1   
6  00cf39c1-Karaptiyan_Robert_50yo_13032021_18590...       0      1        1   
7  00cf39c1-Karaptiyan_Robert_50yo_13032021_18590...       0      1        1   
8  01b0dd74-Gazavandi_Neda_2022-05-14190158_jpg.r...       0      0        1   
9  01b0dd74-Gazavandi_Neda_2022-05-14190158_jpg.r...       0      0        1   

   Implant  Malaligned  Mandibular_Canal  Missing_teeth  Periapical_lesion  \
0        0           0 

In [ ]:
import pandas as pd

# 1. Read the CSV files
features_path = "/content/drive/MyDrive/dental_dataset/combined_features.csv"
df_features = pd.read_csv(features_path)

labels_path = "/content/drive/MyDrive/dental_dataset/image_labels.csv"
df_labels = pd.read_csv(labels_path)

# 2. Clean column names in the labels file
df_labels.columns = df_labels.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')

# 3. Clean image_name columns to ensure proper matching
df_features['image_name'] = df_features['image_name'].str.strip()
df_labels['image_name'] = df_labels['image_name'].str.strip()

# Optionally remove .png extension if needed
df_features['image_name'] = df_features['image_name'].str.replace('.png', '', regex=False)
df_labels['image_name'] = df_labels['image_name'].str.replace('.png', '', regex=False)

# 4. Merge the two DataFrames on image_name
df_combined = pd.merge(df_features, df_labels, on='image_name', how='inner')

# 5. Save the merged DataFrame to a new CSV file
output_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset.csv"
df_combined.to_csv(output_path, index=False)

print(f"Merged successfully! Saved to: {output_path}")
print(f"Rows in final dataset: {df_combined.shape[0]}")
print(f"Columns in final dataset: {df_combined.shape[1]}")


Merged successfully! Saved to: /content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset.csv
Rows in final dataset: 9443
Columns in final dataset: 2085


In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv("/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset.csv")

print("عدد الصفوف:", len(df))
print("الأعمدة الموجودة:")
print(df.columns.tolist())
print("\nأول 10 صفوف:")
print(df.head(10))


عدد الصفوف: 9443
الأعمدة الموجودة:
['image_name', 'cnn_0', 'cnn_1', 'cnn_2', 'cnn_3', 'cnn_4', 'cnn_5', 'cnn_6', 'cnn_7', 'cnn_8', 'cnn_9', 'cnn_10', 'cnn_11', 'cnn_12', 'cnn_13', 'cnn_14', 'cnn_15', 'cnn_16', 'cnn_17', 'cnn_18', 'cnn_19', 'cnn_20', 'cnn_21', 'cnn_22', 'cnn_23', 'cnn_24', 'cnn_25', 'cnn_26', 'cnn_27', 'cnn_28', 'cnn_29', 'cnn_30', 'cnn_31', 'cnn_32', 'cnn_33', 'cnn_34', 'cnn_35', 'cnn_36', 'cnn_37', 'cnn_38', 'cnn_39', 'cnn_40', 'cnn_41', 'cnn_42', 'cnn_43', 'cnn_44', 'cnn_45', 'cnn_46', 'cnn_47', 'cnn_48', 'cnn_49', 'cnn_50', 'cnn_51', 'cnn_52', 'cnn_53', 'cnn_54', 'cnn_55', 'cnn_56', 'cnn_57', 'cnn_58', 'cnn_59', 'cnn_60', 'cnn_61', 'cnn_62', 'cnn_63', 'cnn_64', 'cnn_65', 'cnn_66', 'cnn_67', 'cnn_68', 'cnn_69', 'cnn_70', 'cnn_71', 'cnn_72', 'cnn_73', 'cnn_74', 'cnn_75', 'cnn_76', 'cnn_77', 'cnn_78', 'cnn_79', 'cnn_80', 'cnn_81', 'cnn_82', 'cnn_83', 'cnn_84', 'cnn_85', 'cnn_86', 'cnn_87', 'cnn_88', 'cnn_89', 'cnn_90', 'cnn_91', 'cnn_92', 'cnn_93', 'cnn_94', 'cnn_95', 

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(torch.cuda.current_device()))


True
0
Tesla T4


In [ ]:
# لوضع ال bbox لاحقا

In [ ]:
import json
import pandas as pd
from collections import defaultdict

# Define paths
json_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/annotations/train_coco.json"
csv_path = "/content/drive/MyDrive/dental_dataset/final_dataset_for_training.csv"
output_csv = "/content/drive/MyDrive/dental_dataset/final_dataset_with_bboxes.csv"

# Load JSON annotations
with open(json_path, "r") as f:
    data = json.load(f)

# Create image_id to image info map
image_info = {img["id"]: img for img in data["images"]}
print(f" Loaded {len(image_info)} images from JSON")

# Create filename to image_id map
filename_to_id = {img["file_name"]: img["id"] for img in data["images"]}

# Group annotations by image_id
annotations_by_image = defaultdict(list)
for ann in data["annotations"]:
    image_id = ann["image_id"]
    annotations_by_image[image_id].append(ann)
print(f" Total annotations: {len(data['annotations'])}")

# Read the existing dataset
df = pd.read_csv(csv_path)
print(f"Dataset loaded with {len(df)} rows")

# Number of classes
num_classes = 31

# Prepare bbox columns with initial zeros
bbox_data = {}
for i in range(num_classes):
    bbox_data[f"bbox_{i}_x"] = [0.0] * len(df)
    bbox_data[f"bbox_{i}_y"] = [0.0] * len(df)
    bbox_data[f"bbox_{i}_w"] = [0.0] * len(df)
    bbox_data[f"bbox_{i}_h"] = [0.0] * len(df)

# Create bbox DataFrame and concatenate with main DataFrame
bbox_df = pd.DataFrame(bbox_data)
df = pd.concat([df, bbox_df], axis=1)

# Counter for matched images
updated_count = 0

# Loop over each row and update bbox info
for idx, row in df.iterrows():
    image_name = row["image_name"]

    if image_name not in filename_to_id:
        continue

    image_id = filename_to_id[image_name]

    if image_id not in annotations_by_image:
        continue

    img_w = image_info[image_id]["width"]
    img_h = image_info[image_id]["height"]

    for ann in annotations_by_image[image_id]:
        category_id = ann["category_id"]
        if category_id >= num_classes:
            continue

        x_min, y_min, w, h = ann["bbox"]

        # Convert to relative coords
        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        rel_w = w / img_w
        rel_h = h / img_h

        df.at[idx, f"bbox_{category_id}_x"] = x_center
        df.at[idx, f"bbox_{category_id}_y"] = y_center
        df.at[idx, f"bbox_{category_id}_w"] = rel_w
        df.at[idx, f"bbox_{category_id}_h"] = rel_h

    updated_count += 1

# Save the updated dataset
df.to_csv(output_csv, index=False)
print(f" Successfully added BBox info for {updated_count} images.")
print(f" Updated dataset saved at: {output_csv}")


 Loaded 9481 images from JSON
 Total annotations: 94794
Dataset loaded with 9443 rows
 Successfully added BBox info for 9443 images.
 Updated dataset saved at: /content/drive/MyDrive/dental_dataset/final_dataset_with_bboxes.csv


In [ ]:
# import pandas as pd

# pd.set_option('display.max_columns', None)

# df = pd.read_csv("/content/drive/MyDrive/dental_dataset/final_dataset_with_bboxes.csv")

# print("عدد الصفوف:", len(df))
# print("الأعمدة الموجودة:")
# print(df.columns.tolist())
# print("\nأول 10 صفوف:")
# print(df.head(5))


In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# import torch
# import torch.nn as nn
# from torch.utils.data import TensorDataset, DataLoader
# import torch.optim as optim
# from torch.cuda.amp import autocast, GradScaler
# from sklearn.metrics import f1_score, classification_report

# # ----------------------
# # 1. تحديد الفئات ← الآفات الطبية ← كما هي في ملف CSV
# # ----------------------
# class_names = [
#     'Caries', 'Crown', 'Filling', 'Implant', 'Malaligned',
#     'Mandibular_Canal', 'Missing_teeth', 'Periapical_lesion',
#     'Retained_root', 'Root_Canal_Treatment', 'Root_Piece',
#     'impacted_tooth', 'maxillary_sinus', 'Bone_Loss', 'Fracture_teeth',
#     'Permanent_Teeth', 'Supra_Eruption', 'TAD', 'abutment', 'attrition',
#     'bone_defect', 'gingival_former', 'metal_band', 'orthodontic_brackets',
#     'permanent_retainer', 'post___core', 'plating', 'wire', 'Cyst',
#     'Root_resorption', 'Primary_teeth'
# ]

# num_classes = len(class_names)

# # ----------------------
# # 2. تحديد المسارات ← CSV + الصور
# # ----------------------
# csv_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset.csv"

# print("📌 تحميل البيانات...")
# df = pd.read_csv(csv_path)

# # تحديد الأعمدة ← CNN Features + Manual Features + Labels
# cnn_cols = [col for col in df.columns if col.startswith('cnn_')]
# manual_cols = ['mean_intensity', 'std_intensity', 'dark_pixel_count', 'symmetry_score', 'image_entropy']
# label_cols = class_names

# X_cnn = df[cnn_cols].values.astype(np.float32)
# X_manual = df[manual_cols].values.astype(np.float32)
# y_labels = df[label_cols].values.astype(np.float32)

# print(f"✅ CNN Features Shape: {X_cnn.shape}")
# print(f"✅ Manual Features Shape: {X_manual.shape}")
# print(f"✅ Labels Shape: {y_labels.shape}")

# # ----------------------
# # 3. تقسيم البيانات ← Train / Validation
# # ----------------------
# X_train_cnn, X_val_cnn, X_train_manual, X_val_manual, y_train, y_val = train_test_split(
#     X_cnn, X_manual, y_labels,
#     test_size=0.2,
#     random_state=42
# )

# # ----------------------
# # 4. تحويل البيانات إلى Tensors ← وإنشاء DataLoaders
# # ----------------------
# train_dataset = TensorDataset(
#     torch.tensor(X_train_cnn),
#     torch.tensor(X_train_manual),
#     torch.tensor(y_train)
# )
# val_dataset = TensorDataset(
#     torch.tensor(X_val_cnn),
#     torch.tensor(X_val_manual),
#     torch.tensor(y_val)
# )

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32)

# # ----------------------
# # 5. تعريف النموذج العصبي ← Multi-Label Classification
# # ----------------------
# class DentalClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.classifier = nn.Sequential(
#             nn.Linear(2048 + 5, 1024),
#             nn.ReLU(),
#             nn.BatchNorm1d(1024),
#             nn.Dropout(0.5),

#             nn.Linear(1024, 512),
#             nn.ReLU(),
#             nn.BatchNorm1d(512),
#             nn.Dropout(0.5),

#             nn.Linear(512, num_classes),
#             nn.Sigmoid()
#         )

#     def forward(self, cnn_input, manual_input=None):
#         if manual_input is not None:
#             combined = torch.cat((cnn_input, manual_input), dim=1)
#         else:
#             combined = cnn_input

#         return self.classifier(combined)

# # ----------------------
# # 6. تحديد الجهاز ← CPU / CUDA
# # ----------------------
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = DentalClassifier().to(device)

# # ----------------------
# # 7. حساب Class Weights ← لتحديث الخسارة
# # ----------------------
# pos_weights = []
# for i in range(len(class_names)):
#     neg = (y_train[:, i] == 0).sum()
#     pos = (y_train[:, i] == 1).sum()

#     if pos == 0:
#         print(f"⚠️ الفئة {class_names[i]} ليس لديها أي عينات إيجابية ← سيتم تجاهلها")
#         continue

#     weight = neg / (pos + 1e-6)
#     pos_weights.append(weight)

# pos_weights_tensor = torch.tensor(pos_weights, device=device, dtype=torch.float32)
# criterion = nn.BCELoss()
# optimizer = optim.Adam(model.parameters(), lr=1e-4)

# # ----------------------
# # 8. استخدام GradScaler ← بدون أخطاء ← مع autocast المُعدّلة
# # ----------------------
# scaler = GradScaler()

# # ----------------------
# # 9. Early Stopping ← لتجنب Overfitting
# # ----------------------
# best_val_f1 = 0.0
# patience_counter = 0
# PATIENCE_LIMIT = 15

# # ----------------------
# # 10. حلقة التدريب ← كاملة ← مع الطباعة والتحديث
# # ----------------------
# print("🚀 بدء التدريب...")

# for epoch in range(100):  # عدد Epochs
#     model.train()
#     total_loss = 0.0

#     for batch in train_loader:
#         cnn_batch, manual_batch, label_batch = batch
#         cnn_batch = cnn_batch.to(device)
#         manual_batch = manual_batch.to(device)
#         label_batch = label_batch.to(device)

#         optimizer.zero_grad()

#         with autocast():
#             outputs = model(cnn_batch, manual_batch)
#             loss = criterion(outputs, label_batch)

#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         total_loss += loss.item()

#     # مرحلة Validation
#     model.eval()
#     val_total_loss = 0.0
#     all_preds = []
#     all_true = []

#     with torch.no_grad():
#         for batch in val_loader:
#             cnn_batch, manual_batch, label_batch = batch
#             cnn_batch = cnn_batch.to(device)
#             manual_batch = manual_batch.to(device)
#             label_batch = label_batch.to(device)

#             outputs = model(cnn_batch, manual_batch)
#             val_total_loss += criterion(outputs, label_batch).item()

#             all_preds.append(outputs.cpu())
#             all_true.append(label_batch.cpu())

#     all_preds = torch.cat(all_preds).numpy()
#     all_true = torch.cat(all_true).numpy()

#     all_preds_binary = (all_preds > 0.5).astype(int)
#     all_true_binary = (all_true > 0.5).astype(int)

#     avg_f1 = f1_score(all_true_binary, all_preds_binary, average='samples')

#     print(f"\nEpoch {epoch+1}: Train Loss = {total_loss:.4f}, Val Loss = {val_total_loss:.4f}")
#     print(f"📌 Average F1-Score: {avg_f1:.4f}")

#     # حفظ أفضل نموذج ← باستخدام F1-Score
#     if avg_f1 > best_val_f1:
#         best_val_f1 = avg_f1
#         torch.save(model.state_dict(), "/content/drive/MyDrive/dental_dataset/best_dental_classifier.pt")
#         patience_counter = 0
#         print("✅ تم حفظ النموذج")
#     else:
#         patience_counter += 1
#         print(f"🛑 لم يتم التحسن ← مهلة: {patience_counter}/{PATIENCE_LIMIT}")

#     if patience_counter >= PATIENCE_LIMIT:
#         print("🛑 توقف مبكر ← لا يوجد تحسن.")
#         break

# # ----------------------
# # 11. طباعة تقرير التصنيف النهائي
# # ----------------------
# print("\n📌 التقرير النهائي:")
# print(classification_report(all_true_binary, all_preds_binary, target_names=label_cols, zero_division=0))

📌 تحميل البيانات...
✅ CNN Features Shape: (9443, 2048)
✅ Manual Features Shape: (9443, 5)
✅ Labels Shape: (9443, 31)
⚠️ الفئة bone_defect ليس لديها أي عينات إيجابية ← سيتم تجاهلها
🚀 بدء التدريب...

Epoch 1: Train Loss = 171.9457, Val Loss = 39.6277
📌 Average F1-Score: 0.3303
✅ تم حفظ النموذج

Epoch 2: Train Loss = 157.2935, Val Loss = 34.9228
📌 Average F1-Score: 0.6087
✅ تم حفظ النموذج

Epoch 3: Train Loss = 134.0238, Val Loss = 28.6404
📌 Average F1-Score: 0.6362
✅ تم حفظ النموذج

Epoch 4: Train Loss = 101.6750, Val Loss = 19.8591
📌 Average F1-Score: 0.6703
✅ تم حفظ النموذج

Epoch 5: Train Loss = 73.8400, Val Loss = 14.1641
📌 Average F1-Score: 0.6709
✅ تم حفظ النموذج

Epoch 6: Train Loss = 56.7974, Val Loss = 11.4428
📌 Average F1-Score: 0.6744
✅ تم حفظ النموذج

Epoch 7: Train Loss = 48.4946, Val Loss = 10.4572
📌 Average F1-Score: 0.6687
🛑 لم يتم التحسن ← مهلة: 1/15

Epoch 8: Train Loss = 43.8797, Val Loss = 9.6259
📌 Average F1-Score: 0.6674
🛑 لم يتم التحسن ← مهلة: 2/15

Epoch 9: Train 

In [ ]:
# import cv2
# import os
# from torchvision import transforms
# from PIL import Image
# import torchvision.models as models

# # ----------------------
# # 1. تحويل الصورة ← لتكون مناسبة لـ ResNet50
# # ----------------------
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# # ----------------------
# # 2. استخراج CNN Features ← من الصورة ← باستخدام ResNet50
# # ----------------------
# def extract_cnn_features(image_path):
#     image = Image.open(image_path).convert("RGB")
#     input_tensor = transform(image).unsqueeze(0).to(device)

#     resnet = models.resnet50(pretrained=True)
#     feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).eval().to(device)

#     with torch.no_grad():
#         features = feature_extractor(input_tensor).squeeze().cpu().numpy()

#     return features

# # ----------------------
# # 3. حساب الميزات اليدوية ← من الصورة ← مثل mean_intensity...
# # ----------------------
# def calculate_manual_features(img_path):
#     img = cv2.imread(img_path, 0).astype(np.float32)

#     mean_intensity = np.mean(img)
#     std_intensity = np.std(img)
#     dark_pixel_count = np.sum(img < 50)
#     symmetry_score = abs(np.mean(img[:, :img.shape[1]//2]) - np.mean(img[:, img.shape[1]//2:]))

#     hist = cv2.calcHist([img], [0], None, [256], [0, 256])
#     entropy = -np.sum(hist * np.log2(hist + 1e-7)) / hist.sum()

#     return np.array([mean_intensity, std_intensity, dark_pixel_count, symmetry_score, entropy])

# # ----------------------
# # 4. التنبؤ على صورة جديدة ← بإدخال مسارها فقط
# # ----------------------
# def predict_from_image(image_path, model, class_names):
#     if not os.path.exists(image_path):
#         raise FileNotFoundError(f" لا يمكن العثور على الصورة: {image_path}")

#     model.eval()

#     # 1. استخراج CNN Features ← من الصورة مباشرة
#     cnn_vector = extract_cnn_features(image_path)

#     # 2. حساب الميزات اليدوية ← من الصورة ← بدون CSV
#     manual_vector = calculate_manual_features(image_path)

#     # 3. دمج الميزات ← بنفس الترتيب
#     combined = np.concatenate([cnn_vector, manual_vector]).reshape(1, -1)
#     input_tensor = torch.tensor(combined, dtype=torch.float32).to(device)

#     # 4. التنبؤ
#     with torch.no_grad():
#         output = model(input_tensor).squeeze().cpu().numpy()

#     print("\n الآفات المحتملة في الصورة:")
#     detected = False
#     for idx, prob in enumerate(output):
#         if prob > 0.5:
#             print(f"{class_names[idx]}: {prob:.2f}")
#             detected = True

#     if not detected:
#         print(" لم يتم اكتشاف أي آفة في هذه الصورة.")
# model="/content/drive/MyDrive/dental_dataset/best_dental_classifier.pt"
# # ----------------------
# # 5. تشغيل التنبؤ ← على صورة جديدة ← مثال
# # ----------------------
# predict_from_image(
#     "/content/drive/MyDrive/dental_dataset/dental_dataset/images/test/0e39dd1f-SADRI_NAHID_2020-06-14213242_jpg.rf.f40654b8e66b49d4526718b190c8cc19.jpg",
#     model,
#     class_names
# )

In [5]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os

# ----------------------
# 1. تحميل البيانات ← من CSV ← بدون تقسيم Train/Val
# ----------------------
# csv_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset.csv"
csv_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset_valid.csv"
print(" Loading data from CSV...")
df = pd.read_csv(csv_path)

# ----------------------
# 2. تحديد أعمدة CNN Features ← و Manual Features ← و Labels
# ----------------------
cnn_cols = [col for col in df.columns if col.startswith('cnn_')]
manual_cols = ['mean_intensity', 'std_intensity', 'dark_pixel_count', 'symmetry_score', 'image_entropy']
label_cols = [
    'Caries', 'Crown', 'Filling', 'Implant', 'Malaligned',
    'Mandibular_Canal', 'Missing_teeth', 'Periapical_lesion',
    'Retained_root', 'Root_Canal_Treatment', 'Root_Piece',
    'impacted_tooth', 'maxillary_sinus', 'Bone_Loss', 'Fracture_teeth',
    'Permanent_Teeth', 'Supra_Eruption', 'TAD', 'abutment', 'attrition',
    'bone_defect', 'gingival_former', 'metal_band', 'orthodontic_brackets',
    'permanent_retainer', 'post___core', 'plating', 'wire', 'Cyst',
    'Root_resorption', 'Primary_teeth'
]

# ----------------------
# 3. استخراج CNN Features ← وتطبيع البيانات ← قبل PCA
# ----------------------
X_cnn = df[cnn_cols].values.astype(np.float32)
X_manual = df[manual_cols].values.astype(np.float32)
y_labels = df[label_cols].values.astype(np.float32)

# تنظيف الصفوف التي ليس فيها آفات ← إن كنت قد فعلتها سابقًا
non_zero_rows = y_labels.sum(axis=1) > 0
X_cnn_cleaned = X_cnn[non_zero_rows]
X_manual_cleaned = X_manual[non_zero_rows]
y_cleaned = y_labels[non_zero_rows]

# تطبيع البيانات ← لتحديث PCA
scaler_pca = StandardScaler()
X_cnn_scaled = scaler_pca.fit_transform(X_cnn_cleaned)

# ----------------------
# 4. استخدام PCA ← لتقليل عدد الفيتشرات ← من 2048 → 512 ← أو 1024
# ----------------------
n_components = 512  # يمكنك تغيير هذا الرقم إلى 1024 ← أو أقل ← حسب الحاجة
pca = PCA(n_components=n_components)

# تطبيق PCA ← على CNN Features ← فقط
X_cnn_reduced = pca.fit_transform(X_cnn_scaled)

print(f"\n Explained variance ratio by {n_components} components: {pca.explained_variance_ratio_.sum() * 100:.2f}%")
print(f" New shape after PCA: {X_cnn_reduced.shape}")

# ----------------------
# 5. تحديد الفيتشرات المهمة ← بأسماء الأعمدة ← للاحتفاظ بها ← والباقي حذفه
# ----------------------
# الحصول على أهم الفيتشرات ← من حيث التأثير ← باستخدام Variance ← أو PCA Loadings
explained_variance = pca.explained_variance_ratio_
components_matrix = pca.components_  # شكلها (n_components, n_features)

# تحديد مؤشرات الفيتشرات ذات التأثير الأعلى ← لاستخدامها في CSV
top_feature_indices = np.argsort(np.var(components_matrix, axis=0))[::-1][:n_components]
top_cnn_columns = [cnn_cols[i] for i in top_feature_indices]

# ----------------------
# 6. حفظ قائمة الفيتشرات ← التي يجب الاحتفاظ بها ← لتساعدك على تعديل ملف CSV
# ----------------------
output_dir = "/content/drive/MyDrive/dental_dataset/"
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "important_cnn_features.txt"), "w") as f:
    for col in top_cnn_columns:
        f.write(col + "\n")

print("\n تم حفظ أسماء الفيتشرات المهمة في ملف important_cnn_features.txt")
print(f" يمكنك استخدام هذه الأسماء ← بدل جميع أعمدة cnn_* ← لتحديث ملف CSV ← وتقليل العدد من {len(cnn_cols)} → {n_components}")

 Loading data from CSV...

 Explained variance ratio by 512 components: 97.45%
 New shape after PCA: (2864, 512)

 تم حفظ أسماء الفيتشرات المهمة في ملف important_cnn_features.txt
 يمكنك استخدام هذه الأسماء ← بدل جميع أعمدة cnn_* ← لتحديث ملف CSV ← وتقليل العدد من 2048 → 512


In [6]:
# ----------------------
# 7. إنشاء نسخة جديدة من البيانات ← تحتوي على الفيتشرات المهمة فقط
# ----------------------
selected_cols = top_cnn_columns + manual_cols + label_cols

df_reduced = df[selected_cols]

# حفظ ملف جديد ← يحتوي على الفيتشرات المهمة فقط ← مع اليدوية والعناوين
# reduced_csv_path = "/content/drive/MyDrive/dental_dataset/reduced_final_combined_dataset.csv"
reduced_csv_path = "/content/drive/MyDrive/dental_dataset/reduced_final_combined_dataset_valid.csv"
df_reduced.to_csv(reduced_csv_path, index=False)

print(f"\n تم حفظ ملف جديد ← بصيغة CSV ← يحتوي على {len(top_cnn_columns)} فيتشر CNN مهم ← وجميع الميزات اليدوية والعناوين")
print(f" المسار: {reduced_csv_path}")


 تم حفظ ملف جديد ← بصيغة CSV ← يحتوي على 512 فيتشر CNN مهم ← وجميع الميزات اليدوية والعناوين
 المسار: /content/drive/MyDrive/dental_dataset/reduced_final_combined_dataset_valid.csv


In [7]:
import pandas as pd

pd.set_option('display.max_columns', None)

# df = pd.read_csv("/content/drive/MyDrive/dental_dataset/reduced_final_combined_dataset.csv")
df = pd.read_csv("/content/drive/MyDrive/dental_dataset/reduced_final_combined_dataset_valid.csv")

print("عدد الصفوف:", len(df))
print("الأعمدة الموجودة:")
print(df.columns.tolist())
print("\nأول 10 صفوف:")
print(df.head(10))


عدد الصفوف: 2871
الأعمدة الموجودة:
['cnn_736', 'cnn_1598', 'cnn_452', 'cnn_2040', 'cnn_1697', 'cnn_186', 'cnn_338', 'cnn_41', 'cnn_760', 'cnn_193', 'cnn_644', 'cnn_406', 'cnn_1736', 'cnn_550', 'cnn_1219', 'cnn_427', 'cnn_421', 'cnn_285', 'cnn_612', 'cnn_1006', 'cnn_261', 'cnn_1751', 'cnn_990', 'cnn_1151', 'cnn_1646', 'cnn_54', 'cnn_1888', 'cnn_1196', 'cnn_702', 'cnn_1810', 'cnn_819', 'cnn_986', 'cnn_80', 'cnn_1415', 'cnn_1078', 'cnn_1428', 'cnn_167', 'cnn_989', 'cnn_900', 'cnn_242', 'cnn_1222', 'cnn_884', 'cnn_1756', 'cnn_309', 'cnn_459', 'cnn_46', 'cnn_1819', 'cnn_731', 'cnn_762', 'cnn_1941', 'cnn_1332', 'cnn_1801', 'cnn_1600', 'cnn_1981', 'cnn_1606', 'cnn_725', 'cnn_580', 'cnn_280', 'cnn_1717', 'cnn_792', 'cnn_1592', 'cnn_228', 'cnn_210', 'cnn_25', 'cnn_1546', 'cnn_1121', 'cnn_1988', 'cnn_1147', 'cnn_162', 'cnn_1595', 'cnn_173', 'cnn_1785', 'cnn_1387', 'cnn_1430', 'cnn_425', 'cnn_1064', 'cnn_1994', 'cnn_305', 'cnn_43', 'cnn_754', 'cnn_1571', 'cnn_715', 'cnn_1023', 'cnn_1227', 'cnn_87

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# import torch
# import torch.nn as nn
# from torch.utils.data import TensorDataset, DataLoader
# import torch.optim as optim
# from torch.cuda.amp import autocast, GradScaler
# from sklearn.metrics import f1_score, classification_report
# import os
# from PIL import Image
# from torchvision import transforms
# import cv2


# # ----------------------
# # 1. Load data from CSV
# # ----------------------
# csv_path = "/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset.csv"
# df = pd.read_csv(csv_path)

# print(" Data loaded successfully")

# # Define feature columns
# cnn_cols = [col for col in df.columns if col.startswith('cnn_')]
# manual_cols = ['mean_intensity', 'std_intensity', 'dark_pixel_count', 'symmetry_score', 'image_entropy']
# label_cols = [
#     'Caries', 'Crown', 'Filling', 'Implant', 'Malaligned',
#     'Mandibular_Canal', 'Missing_teeth', 'Periapical_lesion',
#     'Retained_root', 'Root_Canal_Treatment', 'Root_Piece',
#     'impacted_tooth', 'maxillary_sinus', 'Bone_Loss', 'Fracture_teeth',
#     'Permanent_Teeth', 'Supra_Eruption', 'TAD', 'abutment', 'attrition',
#     'bone_defect', 'gingival_former', 'metal_band', 'orthodontic_brackets',
#     'permanent_retainer', 'post___core', 'plating', 'wire', 'Cyst',
#     'Root_resorption', 'Primary_teeth'
# ]

# # Extract features and labels
# X_cnn = df[cnn_cols].values.astype(np.float32)
# X_manual = df[manual_cols].values.astype(np.float32)
# y_labels = df[label_cols].values.astype(np.float32)

# print(f" CNN Features Shape: {X_cnn.shape}")
# print(f" Manual Features Shape: {X_manual.shape}")
# print(f" Labels Shape: {y_labels.shape}")

# # ----------------------
# # 2. Remove rows with no lesions (all zeros in labels)
# # ----------------------
# non_zero_rows = y_labels.sum(axis=1) > 0
# X_cnn_cleaned = X_cnn[non_zero_rows]
# X_manual_cleaned = X_manual[non_zero_rows]
# y_cleaned = y_labels[non_zero_rows]

# print(f"\n Rows after cleaning: {len(X_cnn_cleaned)} / {len(X_cnn)}")

# # ----------------------
# # 3. Split data into Train / Validation
# # ----------------------
# X_train_cnn, X_val_cnn, X_train_manual, X_val_manual, y_train, y_val = train_test_split(
#     X_cnn_cleaned, X_manual_cleaned, y_cleaned,
#     test_size=0.2,
#     random_state=42
# )

# # Convert to Tensors
# train_dataset = TensorDataset(
#     torch.tensor(X_train_cnn),
#     torch.tensor(X_train_manual),
#     torch.tensor(y_train)
# )
# val_dataset = TensorDataset(
#     torch.tensor(X_val_cnn),
#     torch.tensor(X_val_manual),
#     torch.tensor(y_val)
# )

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32)

# # ----------------------
# # 4. Build the model
# # ----------------------
# class DentalClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.classifier = nn.Sequential(
#             nn.Linear(2048 + 5, 1024),
#             nn.ReLU(),
#             nn.BatchNorm1d(1024),
#             nn.Dropout(0.5),

#             nn.Linear(1024, 512),
#             nn.ReLU(),
#             nn.BatchNorm1d(512),
#             nn.Dropout(0.5),

#             nn.Linear(512, len(label_cols)),
#             nn.Sigmoid()
#         )

#     def forward(self, cnn_input, manual_input=None):
#         if manual_input is not None:
#             combined = torch.cat((cnn_input, manual_input), dim=1)
#         else:
#             combined = cnn_input

#         return self.classifier(combined)

# device = "cuda" if torch.cuda.is_available() else "cpu"

# model = DentalClassifier().to(device)
# optimizer = optim.Adam(model.parameters(), lr=1e-4)

# # ----------------------
# # 5. Compute Class Weights
# # ----------------------
# pos_weights = []
# for i in range(len(label_cols)):
#     neg = (y_train[:, i] == 0).sum()
#     pos = (y_train[:, i] == 1).sum()

#     if pos == 0:
#         print(f" Class {label_cols[i]} has no positive samples")
#         continue

#     weight = neg / (pos + 1e-6)
#     pos_weights.append(weight)

# pos_weights_tensor = torch.tensor(pos_weights, device=device, dtype=torch.float32)
# criterion = nn.BCELoss()

# # ----------------------
# # 6. Use GradScaler for mixed precision training
# # ----------------------
# scaler = GradScaler()

# # ----------------------
# # 7. Early Stopping setup
# # ----------------------
# best_val_f1 = 0.0
# patience_counter = 0
# PATIENCE_LIMIT = 15

# # ----------------------
# # 8. Training loop
# # ----------------------
# print(" Starting training...")

# for epoch in range(100):  # Max Epochs
#     model.train()
#     total_loss = 0.0

#     for batch in train_loader:
#         cnn_batch, manual_batch, label_batch = batch
#         cnn_batch = cnn_batch.to(device)
#         manual_batch = manual_batch.to(device)
#         label_batch = label_batch.to(device)

#         optimizer.zero_grad()

#         with autocast():
#             outputs = model(cnn_batch, manual_batch)
#             loss = criterion(outputs, label_batch)

#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         total_loss += loss.item()

#     # Validation phase
#     model.eval()
#     val_total_loss = 0.0
#     all_preds = []
#     all_true = []

#     with torch.no_grad():
#         for batch in val_loader:
#             cnn_batch, manual_batch, label_batch = batch
#             cnn_batch = cnn_batch.to(device)
#             manual_batch = manual_batch.to(device)
#             label_batch = label_batch.to(device)

#             outputs = model(cnn_batch, manual_batch)
#             val_total_loss += criterion(outputs, label_batch).item()

#             all_preds.append(outputs.cpu())
#             all_true.append(label_batch.cpu())

#     all_preds = torch.cat(all_preds).numpy()
#     all_true = torch.cat(all_true).numpy()

#     # Threshold predictions
#     all_preds_binary = (all_preds > 0.5).astype(int)
#     all_true_binary = (all_true > 0.5).astype(int)

#     avg_f1 = f1_score(all_true_binary, all_preds_binary, average='samples')

#     print(f"\nEpoch {epoch+1}: Train Loss = {total_loss:.4f}, Val Loss = {val_total_loss:.4f}")
#     print(f" Average F1-Score: {avg_f1:.4f}")

#     # Save best model
#     if avg_f1 > best_val_f1:
#         best_val_f1 = avg_f1
#         torch.save(model.state_dict(), "/content/drive/MyDrive/dental_dataset/best_dental_classifier.pt")
#         patience_counter = 0
#         print(" Best model saved")
#     else:
#         patience_counter += 1
#         print(f" No improvement. Patience: {patience_counter}/{PATIENCE_LIMIT}")

#     if patience_counter >= PATIENCE_LIMIT:
#         print(" Training stopped early due to no improvement.")
#         break

# # ----------------------
# # 9. Final evaluation report
# # ----------------------
# print("\n Final Classification Report:")
# print(classification_report(all_true_binary, all_preds_binary, target_names=label_cols, zero_division=0))

 Data loaded successfully
 CNN Features Shape: (9443, 2048)
 Manual Features Shape: (9443, 5)
 Labels Shape: (9443, 31)

 Rows after cleaning: 9443 / 9443
 Class bone_defect has no positive samples
 Starting training...


<ipython-input-6-3861845694>:136: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.11/dist-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
<ipython-input-6-3861845694>:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(



Epoch 1: Train Loss = 171.3982, Val Loss = 38.7593
 Average F1-Score: 0.3362
 Best model saved


<ipython-input-6-3861845694>:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(



Epoch 2: Train Loss = 156.6761, Val Loss = 35.1003
 Average F1-Score: 0.6100
 Best model saved


...


Epoch 61: Train Loss = 30.5009, Val Loss = 7.7648
 Average F1-Score: 0.7206
 No improvement. Patience: 13/15


<ipython-input-6-3861845694>:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(



Epoch 62: Train Loss = 30.3859, Val Loss = 7.3569
 Average F1-Score: 0.7293
 No improvement. Patience: 14/15


<ipython-input-6-3861845694>:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(



Epoch 63: Train Loss = 30.8486, Val Loss = 7.4362
 Average F1-Score: 0.7196
 No improvement. Patience: 15/15
 Training stopped early due to no improvement.

 Final Classification Report:
                      precision    recall  f1-score   support

              Caries       0.81      0.55      0.66       433
               Crown       0.68      0.45      0.54       566
             Filling       0.75      0.96      0.84      1361
             Implant       0.71      0.15      0.24       103
          Malaligned       0.00      0.00      0.00         3
    Mandibular_Canal       0.00      0.00      0.00        47
       Missing_teeth       0.74      0.45      0.56       231
   Periapical_lesion       0.57      0.01      0.02       340
       Retained_root       0.00      0.00      0.00         8
Root_Canal_Treatment       0.72      0.60      0.65       750
          Root_Piece       1.00      0.05      0.09       132
      impacted_tooth       0.94      0.96      0.95      1545
     

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report
import os
import matplotlib.pyplot as plt

# ----------------------
# 1. تحديد الجهاز ← CPU / GPU
# ----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

# ----------------------
# 2. تحميل بيانات التدريب ← و Val من ملفين مختلفين
# ----------------------
train_csv = "/content/drive/MyDrive/dental_dataset/reduced_final_combined_dataset.csv"
val_csv = "/content/drive/MyDrive/dental_dataset/dental_dataset/final_combined_dataset_valid.csv"

df_train = pd.read_csv(train_csv)
df_val = pd.read_csv(val_csv)

# ----------------------
# 3. تحديد الأعمدة ← CNN + Manual Features + Labels
# ----------------------
cnn_cols = [col for col in df_train.columns if col.startswith('cnn_')]
manual_cols = ['mean_intensity', 'std_intensity', 'dark_pixel_count', 'symmetry_score', 'image_entropy']
label_cols = [
    'Caries', 'Crown', 'Filling', 'Implant', 'Malaligned',
    'Mandibular_Canal', 'Missing_teeth', 'Periapical_lesion',
    'Retained_root', 'Root_Canal_Treatment', 'Root_Piece',
    'impacted_tooth', 'maxillary_sinus', 'Bone_Loss', 'Fracture_teeth',
    'Permanent_Teeth', 'Supra_Eruption', 'TAD', 'abutment', 'attrition',
    'bone_defect', 'gingival_former', 'metal_band', 'orthodontic_brackets',
    'permanent_retainer', 'post___core', 'plating', 'wire', 'Cyst',
    'Root_resorption', 'Primary_teeth'
]

# ----------------------
# 4. استخراج الميزات والعناوين ← من Train و Val
# ----------------------
X_cnn_train = df_train[cnn_cols].values.astype(np.float32)
X_manual_train = df_train[manual_cols].values.astype(np.float32)
y_train = df_train[label_cols].values.astype(np.float32)

X_cnn_val = df_val[cnn_cols].values.astype(np.float32)
X_manual_val = df_val[manual_cols].values.astype(np.float32)
y_val = df_val[label_cols].values.astype(np.float32)

# ----------------------
# 5. تنظيف الصفوف التي ليس فيها أي آفة ← في Train فقط
# ----------------------
non_zero_rows_train = y_train.sum(axis=1) > 0
X_cnn_train_cleaned = X_cnn_train[non_zero_rows_train]
X_manual_train_cleaned = X_manual_train[non_zero_rows_train]
y_train_cleaned = y_train[non_zero_rows_train]

# ----------------------
# 6. تطبيع البيانات ← لتحديث الاستقرار العددي
# ----------------------
from sklearn.preprocessing import StandardScaler

scaler_cnn = StandardScaler()
X_cnn_train_scaled = scaler_cnn.fit_transform(X_cnn_train_cleaned)
X_cnn_val_scaled = scaler_cnn.transform(X_cnn_val)

scaler_manual = StandardScaler()
X_manual_train_scaled = scaler_manual.fit_transform(X_manual_train_cleaned)
X_manual_val_scaled = scaler_manual.transform(X_manual_val)

# ----------------------
# 7. بناء Tensor Dataset ← بدون تقسيم Train/Val
# ----------------------
train_dataset = TensorDataset(
    torch.tensor(X_cnn_train_scaled),
    torch.tensor(X_manual_train_scaled),
    torch.tensor(y_train_cleaned)
)
val_dataset = TensorDataset(
    torch.tensor(X_cnn_val_scaled),
    torch.tensor(X_manual_val_scaled),
    torch.tensor(y_val)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

num_classes = len(label_cols)
print(f"[INFO] Number of classes: {num_classes}")
print(f"[INFO] Train Shape: {X_cnn_train_scaled.shape}, Val Shape: {X_cnn_val_scaled.shape}")

# ----------------------
# 8. تعريف النموذج ← بدون Sigmoid ← لأننا نستخدم BCEWithLogitsLoss
# ----------------------
class DentalClassifier(nn.Module):
    def __init__(self, input_dim=517):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),

            nn.Linear(512, num_classes)
        )

    def forward(self, cnn_input, manual_input=None):
        if manual_input is not None:
            combined = torch.cat((cnn_input, manual_input), dim=1)
        else:
            combined = cnn_input

        return self.classifier(combined)

model = DentalClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = GradScaler()

# ----------------------
# 9. حساب Class Weights ← لتحديث الخسارة ← للفئات النادرة
# ----------------------
pos_weights = []
for i in range(num_classes):
    neg = (y_train_cleaned[:, i] == 0).sum()
    pos = (y_train_cleaned[:, i] == 1).sum()

    if pos == 0:
        print(f"[WARNING] Class {label_cols[i]} has no positive samples")
        pos_weights.append(1.0)
    else:
        weight = neg / (pos + 1e-6)
        pos_weights.append(weight)

pos_weights_tensor = torch.tensor(pos_weights, device=device, dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)

# ----------------------
# 10. Early Stopping ← و LR Scheduler ← لتحديث الأداء
# ----------------------
best_val_f1 = 0.0
patience_counter = 0
PATIENCE_LIMIT = 20

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

# ----------------------
# 11. قوائم لحفظ القيم ← لرسم المنحنى بعد التدريب
# ----------------------
train_losses = []
val_losses = []
train_f1s = []
val_f1s = []

# ----------------------
# 12. حلقة التدريب ← مع استخدام Val من ملف CSV ← وليس تقسيم عشوائي
# ----------------------
print("[INFO] Starting training...")

all_preds = []
all_true = []

for epoch in range(200):  # ↑ تم زيادة Epochs ← لتمنح النموذج فرصة أكبر
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        cnn_batch, manual_batch, label_batch = batch
        cnn_batch = cnn_batch.to(device)
        manual_batch = manual_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad()

        with autocast("cuda", enabled=device == "cuda"):
            outputs = model(cnn_batch, manual_batch)
            loss = criterion(outputs, label_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    # مرحلة التحقق ← بدون تحديث الأوزان ← مع حساب F1-Score
    model.eval()
    val_total_loss = 0.0
    all_val_preds = []
    all_val_true = []

    with torch.no_grad():
        for batch in val_loader:
            cnn_batch, manual_batch, label_batch = batch
            cnn_batch = cnn_batch.to(device)
            manual_batch = manual_batch.to(device)
            label_batch = label_batch.to(device)

            outputs = model(cnn_batch, manual_batch)
            val_total_loss += criterion(outputs, label_batch).item()

            all_val_preds.append(torch.sigmoid(outputs).cpu())
            all_val_true.append(label_batch.cpu())

    all_val_preds_np = torch.cat(all_val_preds).numpy()
    all_val_true_np = torch.cat(all_val_true).numpy()

    all_val_preds_binary = (all_val_preds_np > 0.5).astype(int)
    all_val_true_binary = (all_val_true_np > 0.5).astype(int)

    avg_f1 = f1_score(all_val_true_binary, all_val_preds_binary, average='samples')

    print(f"\n[Epoch {epoch+1}] Train Loss = {total_loss:.4f}, Val F1-Score = {avg_f1:.4f}")

    # حفظ أفضل نموذج ← فقط إن كان هناك تحسن
    if avg_f1 > best_val_f1:
        best_val_f1 = avg_f1
        torch.save(model.state_dict(), "/content/drive/MyDrive/dental_dataset/best_dental_model_with_external_val.pt")
        patience_counter = 0
        print(" Best model saved.")
    else:
        patience_counter += 1
        print(f" No improvement. Patience: {patience_counter}/{PATIENCE_LIMIT}")

    # تحديث LR ← عند عدم التحسن ← لتحديث الأداء
    scheduler.step(avg_f1)

    # إضافة القيم إلى القوائم ← لرسم المنحنى
    train_losses.append(total_loss)
    val_losses.append(val_total_loss)
    train_f1s.append(avg_f1)
    val_f1s.append(avg_f1)

    # التوقف المبكر ← عند عدم التحسن منذ فترة طويلة
    if patience_counter >= PATIENCE_LIMIT:
        print(" Training stopped early due to no improvement.")
        break

# ----------------------
# 13. رسم المنحنى ← لتحديد Overfitting أو Underfitting
# ----------------------
print("\n Plotting training curves...")

plt.figure(figsize=(12, 5))

# Loss Curve
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss", color="blue")
plt.plot(val_losses, label="Val Loss", color="orange")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# F1-Score Curve
plt.subplot(1, 2, 2)
plt.plot(train_f1s, label="Train F1-Score", color="green")
plt.plot(val_f1s, label="Val F1-Score", color="red")
plt.title("F1-Score Curve")
plt.xlabel("Epoch")
plt.ylabel("F1-Score")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ----------------------
# 14. طباعة التقرير النهائي ← بعد انتهاء التدريب
# ----------------------
print("\n Final Classification Report:")
print(classification_report(all_val_true_binary, all_val_preds_binary, target_names=label_cols, zero_division=0))

ImportError: cannot import name 'PCA' from 'sklearn.preprocessing' (/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/__init__.py)

In [17]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import models
from PIL import Image
import cv2
import numpy as np
import os

# ----------------------
# 1. تحديد الفئات ← كما هي في ملف CSV الخاص بك
# ----------------------
class_names = [
    'Caries', 'Crown', 'Filling', 'Implant', 'Malaligned',
    'Mandibular_Canal', 'Missing_teeth', 'Periapical_lesion',
    'Retained_root', 'Root_Canal_Treatment', 'Root_Piece',
    'impacted_tooth', 'maxillary_sinus', 'Bone_Loss', 'Fracture_teeth',
    'Permanent_Teeth', 'Supra_Eruption', 'TAD', 'abutment', 'attrition',
    'bone_defect', 'gingival_former', 'metal_band', 'orthodontic_brackets',
    'permanent_retainer', 'post___core', 'plating', 'wire', 'Cyst',
    'Root_resorption', 'Primary_teeth'
]

num_classes = len(class_names)

# ----------------------
# 2. تحديد المسار ← للنموذج المحفوظ
# ----------------------
model_path = "/content/drive/MyDrive/dental_dataset/best_dental_classifier.pt"

# ----------------------
# 3. تعريف طبقة النموذج ← بنفس الطريقة مثل مرحلة التدريب
# ----------------------
class DentalClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(2048 + 5, 1024),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(1024),
            torch.nn.Dropout(0.5),

            torch.nn.Linear(1024, 512),
            torch.nn.ReLU(),
            torch.nn.BatchNorm1d(512),
            torch.nn.Dropout(0.5),

            torch.nn.Linear(512, num_classes),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(x)

In [11]:
# ----------------------
# 4. تحويل الصورة ← لتكون مناسبة لـ ResNet50
# ----------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ----------------------
# 5. استخراج CNN Features ← من الصورة ← باستخدام ResNet50
# ----------------------
def extract_cnn_features(image_path):
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    resnet = models.resnet50(pretrained=True)
    feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1]).eval().to(device)

    with torch.no_grad():
        features = feature_extractor(input_tensor).squeeze().cpu().numpy()

    return features

# ----------------------
# 6. حساب الميزات اليدوية ← من الصورة ← بدون CSV
# ----------------------
def calculate_manual_features(img_path):
    img = cv2.imread(img_path, 0).astype(np.float32)

    mean_intensity = np.mean(img)
    std_intensity = np.std(img)
    dark_pixel_count = np.sum(img < 50)
    symmetry_score = abs(np.mean(img[:, :img.shape[1]//2]) - np.mean(img[:, img.shape[1]//2:]))

    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    entropy = -np.sum(hist * np.log2(hist + 1e-7)) / hist.sum()

    return np.array([mean_intensity, std_intensity, dark_pixel_count, symmetry_score, entropy])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------
# 7. تحميل النموذج ← بصيغة .pt ← بدون إعادة التدريب
# ----------------------
print(" Loading model...")
model = DentalClassifier().to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

# ----------------------
# 8. دالة التنبؤ ← بإدخال الصورة فقط ← بدون تغيير
# ----------------------
def predict_from_image(image_path, model, class_names):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f" Image file not found: {image_path}")

    model.eval()

    # 1. استخراج CNN Features من الصورة
    cnn_vector = extract_cnn_features(image_path)

    # 2. حساب الميزات اليدوية ← من الصورة مباشرة
    manual_vector = calculate_manual_features(image_path)

    # 3. دمج الميزات ← بنفس الترتيب
    combined = np.concatenate([cnn_vector, manual_vector]).reshape(1, -1)
    input_tensor = torch.tensor(combined, dtype=torch.float32).to(device)

    # 4. التنبؤ
    with torch.no_grad():
        output = model(input_tensor).squeeze().cpu().numpy()

    print("\n Lesions detected in the image:")
    detected = False
    for idx, prob in enumerate(output):
        if prob > 0.5:
            print(f"{class_names[idx]}: {prob:.2f}")
            detected = True

    if not detected:
        print(" No lesions detected.")

 Loading model...


In [ ]:
predict_from_image(
    "/content/drive/MyDrive/dental_dataset/dental_dataset/images/test/796070000-jpg_png_jpg.rf.703b86e988330cb2310bfe4c271551a1.jpg",
    model,
    class_names
)


 Lesions detected in the image:
Filling: 0.72
impacted_tooth: 0.96
